# 20 - Incremental Gold Funnel with CDF

Reads only unprocessed Delta Change Data Feed versions from
`silver_events`, derives every affected `event_date` from both
preimages and postimages, then recomputes and merges only those
dates into `gold_daily_funnel`.

The Delta-version watermark advances only after a successful
update. Therefore a seven-day late event updates its historical
business date without requiring a full Gold rebuild.


In [ ]:
%run ./05_bronze_common


In [ ]:
import uuid

from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F


dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")

catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
job_run_id = dbutils.widgets.get("job_run_id").strip()
spark.conf.set("spark.sql.session.timeZone", "UTC")

pipeline_name = "gold_daily_funnel_cdf"
watermark_name = "silver_events_cdf_version"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
source_table = f"{catalog}.silver.silver_events"
target_table = f"{catalog}.gold.gold_daily_funnel"
watermark_table = f"{catalog}.control.pipeline_watermarks"
lineage_table = f"{catalog}.control.record_lineage"

if not spark.catalog.tableExists(source_table):
    raise RuntimeError(f"Run Silver first; missing {source_table}")
if not spark.catalog.tableExists(target_table):
    raise RuntimeError(f"Run 15_gold_daily_funnel first; missing {target_table}")
source_detail = spark.sql(f"DESCRIBE DETAIL {source_table}").first()
if source_detail["properties"].get("delta.enableChangeDataFeed") != "true":
    raise RuntimeError(
        f"CDF is not enabled on {source_table}; rerun 10_silver_events "
        "after enabling delta.enableChangeDataFeed"
    )


def recompute_funnel(events):
    events = (
        events.withColumn(
            "actor_id",
            F.coalesce(
                F.col("user_id"),
                F.concat(F.lit("anonymous:"), F.col("session_id")),
            ),
        )
        .filter(F.col("actor_id").isNotNull())
    )
    first_views = (
        events.filter(F.col("event_type") == "view")
        .groupBy("event_date", "actor_id")
        .agg(F.min("event_time").alias("first_view_ts"))
    )
    first_adds = (
        events.filter(F.col("event_type") == "add_to_cart")
        .join(first_views, ["event_date", "actor_id"], "inner")
        .filter(F.col("event_time") >= F.col("first_view_ts"))
        .groupBy("event_date", "actor_id", "first_view_ts")
        .agg(F.min("event_time").alias("first_add_to_cart_ts"))
    )
    first_purchases = (
        events.filter(F.col("event_type") == "purchase")
        .join(
            first_adds.select(
                "event_date", "actor_id", "first_add_to_cart_ts"
            ),
            ["event_date", "actor_id"],
            "inner",
        )
        .filter(F.col("event_time") >= F.col("first_add_to_cart_ts"))
        .groupBy("event_date", "actor_id")
        .agg(F.min("event_time").alias("first_purchase_ts"))
    )
    actor_funnel = (
        first_views.join(
            first_adds.select(
                "event_date", "actor_id", "first_add_to_cart_ts"
            ),
            ["event_date", "actor_id"],
            "left",
        )
        .join(first_purchases, ["event_date", "actor_id"], "left")
    )
    return (
        actor_funnel.groupBy("event_date")
        .agg(
            F.countDistinct("actor_id").cast("long").alias("view_users"),
            F.countDistinct(
                F.when(
                    F.col("first_add_to_cart_ts").isNotNull(),
                    F.col("actor_id"),
                )
            ).cast("long").alias("add_to_cart_users"),
            F.countDistinct(
                F.when(
                    F.col("first_purchase_ts").isNotNull(),
                    F.col("actor_id"),
                )
            ).cast("long").alias("purchase_users"),
        )
        .withColumn(
            "record_hash",
            F.sha2(
                F.concat_ws(
                    "||",
                    "view_users",
                    "add_to_cart_users",
                    "purchase_users",
                ),
                256,
            ),
        )
        .withColumn("updated_ts", F.current_timestamp())
        .withColumn("pipeline_run_id", F.lit(pipeline_run_id))
    )


def update_version_watermark(version):
    source = spark.range(1).select(
        F.lit(pipeline_name).alias("pipeline_name"),
        F.lit(watermark_name).alias("watermark_name"),
        F.current_timestamp().alias("watermark_ts"),
        F.lit(str(version)).alias("watermark_value"),
        F.current_timestamp().alias("updated_ts"),
        F.lit(pipeline_run_id).alias("pipeline_run_id"),
    )
    (
        DeltaTable.forName(spark, watermark_table)
        .alias("target")
        .merge(
            source.alias("source"),
            """
            target.pipeline_name = source.pipeline_name
            AND target.watermark_name = source.watermark_name
            """,
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


In [ ]:
history = spark.sql(f"DESCRIBE HISTORY {source_table}")
current_version = int(history.agg(F.max("version").alias("version")).first().version)
watermark = (
    spark.table(watermark_table)
    .filter(
        (F.col("pipeline_name") == pipeline_name)
        & (F.col("watermark_name") == watermark_name)
    )
    .orderBy(F.col("updated_ts").desc())
    .first()
)
last_version = int(watermark.watermark_value) if watermark else -1
starting_version = last_version + 1
batch_id = f"versions_{starting_version}_to_{current_version}"

upsert_audit(
    catalog, pipeline_run_id, pipeline_name, batch_id, start_ts,
    "RUNNING", [source_table], job_run_id,
)

try:
    if starting_version > current_version:
        upsert_audit(
            catalog, pipeline_run_id, pipeline_name, batch_id, start_ts,
            "SUCCESS", [source_table], job_run_id,
            end_ts=datetime.now(timezone.utc), rows_read=0,
            rows_written=0, rows_quarantined=0,
        )
        print(
            f"No unprocessed CDF versions. Watermark remains at {last_version}."
        )
    else:
        changes = (
            spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", starting_version)
            .option("endingVersion", current_version)
            .table(source_table)
            .cache()
        )
        changed_rows = changes.count()
        affected_dates = (
            changes.filter(F.col("event_date").isNotNull())
            .select("event_date")
            .distinct()
            .cache()
        )
        affected_date_count = affected_dates.count()

        rows_written = 0
        if affected_date_count:
            current_events = (
                spark.table(source_table)
                .join(affected_dates, "event_date", "inner")
                .cache()
            )
            recomputed = recompute_funnel(current_events).cache()
            rows_written = recomputed.count()

            stale_dates = (
                spark.table(target_table)
                .select("event_date")
                .join(affected_dates, "event_date", "inner")
                .join(recomputed.select("event_date"), "event_date", "left_anti")
                .withColumn("_delete", F.lit(True))
            )
            merge_source = (
                recomputed.withColumn("_delete", F.lit(False))
                .unionByName(stale_dates, allowMissingColumns=True)
            )
            assignments = {
                column: f"source.`{column}`" for column in recomputed.columns
            }
            (
                DeltaTable.forName(spark, target_table)
                .alias("target")
                .merge(
                    merge_source.alias("source"),
                    "target.event_date = source.event_date",
                )
                .whenMatchedDelete(condition="source._delete")
                .whenMatchedUpdate(
                    condition=(
                        "NOT source._delete "
                        "AND NOT (target.record_hash <=> source.record_hash)"
                    ),
                    set=assignments,
                )
                .whenNotMatchedInsert(
                    condition="NOT source._delete",
                    values=assignments,
                )
                .execute()
            )

            lineage_keys = affected_dates.select(
                F.col("event_date").cast("string").alias("target_record_key")
            )
            (
                DeltaTable.forName(spark, lineage_table)
                .alias("target")
                .merge(
                    lineage_keys.alias("source"),
                    f"""
                    target.target_table = '{target_table}'
                    AND target.target_record_key = source.target_record_key
                    """,
                )
                .whenMatchedDelete()
                .execute()
            )
            lineage = current_events.select(
                F.lit(target_table).alias("target_table"),
                F.col("event_date").cast("string").alias("target_record_key"),
                F.lit(pipeline_run_id).alias("pipeline_run_id"),
                F.lit(source_table).alias("source_table"),
                F.col("event_id").alias("source_record_key"),
                "source_file",
                "batch_id",
                F.col("ingest_ts").alias("source_ingest_ts"),
                F.current_timestamp().alias("recorded_ts"),
            ).dropDuplicates(
                [
                    "target_table", "target_record_key", "source_table",
                    "source_record_key", "source_file", "batch_id",
                ]
            )
            if lineage.limit(1).count():
                lineage.write.format("delta").mode("append").saveAsTable(
                    lineage_table
                )

        update_version_watermark(current_version)
        upsert_audit(
            catalog, pipeline_run_id, pipeline_name, batch_id, start_ts,
            "SUCCESS", [source_table], job_run_id,
            end_ts=datetime.now(timezone.utc), rows_read=changed_rows,
            rows_written=rows_written, rows_quarantined=0,
        )
        print(
            f"Processed CDF versions {starting_version}-{current_version}; "
            f"changed_rows={changed_rows}, affected_dates={affected_date_count}."
        )
except Exception as exc:
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, batch_id, start_ts,
        "FAILED", [source_table], job_run_id,
        end_ts=datetime.now(timezone.utc), error_message=str(exc)[:4000],
    )
    raise

spark.table(target_table).orderBy("event_date").display()
